# 🗒️ Timetable Optimization Verification Notebook
This lightweight standalone notebook runs the **Google OR-Tools CP-SAT** optimization algorithm locally to verify correctness and visualize the generated multi-constraint schedule directly using your mom's real-world extracted configuration data.

In [1]:
import json
import os
import sys

# Ensure local backend packages resolve correctly
sys.path.append(os.path.abspath('.'))

from backend.optimizer.engine import generate_timetable

# Load extracted real configuration dataset
data_path = os.path.join('config', 'my_school_config.json')
with open(data_path, 'r') as f:
    school_data = json.load(f)

print(f"Loaded {len(school_data['teachers'])} teachers, {len(school_data['classes'])} classes, and {len(school_data['subjects'])} subjects successfully from real configuration.")

Loaded 90 teachers, 56 classes, and 41 subjects successfully from real configuration.


In [2]:
# Execute the global CP-SAT solver engine
print('Solving multi-dimensional timetable constraints...')
result = generate_timetable(school_data)

print(f"Optimization Success: {result['success']}")
print(f"Solver Status: {result['status']}")
print(f"Total Assignments Computed: {len(result.get('assignments', []))}")
print(f"Convergence Duration: {result.get('metadata', {}).get('solve_duration_seconds', 0)} seconds")

Solving multi-dimensional timetable constraints...
Optimization Success: True
Solver Status: OPTIMAL
Total Assignments Computed: 1287
Convergence Duration: 3.77 seconds


In [3]:
# Verify and output the full weekly schedule for the first Class group
if result['success'] and school_data['classes']:
    target_class = school_data['classes'][0]
    c_id = target_class['id']
    c_name = target_class['name']
    
    assignments = result['assignments']
    class_assignments = [a for a in assignments if a['class_id'] == c_id]
    
    schedule = {}
    for day in school_data['days']:
        schedule[day] = ['Free Slot'] * school_data['periods_per_day']
        
    for a in class_assignments:
        day = a['day']
        p_idx = a['period'] - 1
        sub_name = next((s['name'] for s in school_data['subjects'] if s['id'] == a['subject_id']), a['subject_id'])
        t_name = next((t['name'] for t in school_data['teachers'] if t['id'] == a['teacher_id']), a['teacher_id'])
        
        # Extract short recognizable label
        short_teacher = t_name.split(' ')[0] if ' ' in t_name else t_name[:10]
        schedule[day][p_idx] = f"{sub_name[:10]} ({short_teacher})"
        
    print('\n' + '='*115)
    print(' '*40 + f"🗓️ CLASS {c_name} VERIFIED TIMETABLE")
    print('='*115)
    
    # Print header row
    header = "| {:<12} |".format("Period / Day")
    for d in school_data['days']:
        header += " {:<14} |".format(d)
    print(header)
    print('-'*len(header))
    
    # Print period rows
    for p in range(school_data['periods_per_day']):
        row = "| {:<12} |".format(f"Period {p+1}")
        for d in school_data['days']:
            val = schedule[d][p]
            row += " {:<14} |".format(val)
        print(row)
    print('='*115)
else:
    print('⚠️ Optimization Failed. Details:', result.get('violations'))


                                        🗓️ CLASS 1 - A VERIFIED TIMETABLE
| Period / Day | Monday         | Tuesday        | Wednesday      | Thursday       | Friday         | Saturday       |
----------------------------------------------------------------------------------------------------------------------
| Period 1     | COMPUTERS (PREETI) | ENG (PREETI)   | Free Slot      | ENG (PREETI)   | Free Slot      | Free Slot      |
| Period 2     | CLASS TEAC (PREETI) | HINDI (POOJA)  | Free Slot      | MATHS (PREETI) | Free Slot      | MATHS (PREETI) |
| Period 3     | DANCE (NIDHI)  | Free Slot      | Free Slot      | Free Slot      | Free Slot      | ART (PREETI)   |
| Period 4     | Free Slot      | HINDI (POOJA)  | Free Slot      | Free Slot      | Free Slot      | MATHS (PREETI) |
| Period 5     | PHY EDU (VEENA) | GK (PREETI)    | STORY TIME (PREETI) | Free Slot      | EVS (PREETI)   | Free Slot      |
| Period 6     | Free Slot      | Free Slot      | ENG (PREETI)   | MATHS (PR